# Intermediate 12 — Bayesian Inference Basics

Practice notebook for the **"Bayesian Inference Basics"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Bayes' rule for parameters

In the Bayesian framework the parameter \(\theta\) is a **random variable**
with prior density \(\pi(\theta)\). Given data \(x\), the **posterior** density
is

$$
\pi(\theta \mid x) =
\frac{f(x \mid \theta)\,\pi(\theta)}
     {\int f(x \mid \theta')\,\pi(\theta')\,d\theta'}.
$$

- **Numerator** \(f(x\mid\theta)\,\pi(\theta)\): the *unnormalized posterior*
  (likelihood \(\times\) prior).
- **Denominator** \(\int f(x\mid\theta')\,\pi(\theta')\,d\theta'\): the
  *marginal likelihood* \(f(x)\), a constant in \(\theta\) that normalizes the
  posterior so it integrates to \(1\).

The key idea: the posterior is the prior **reweighted** by how well each
\(\theta\) explains the data. We will *compute* it numerically on a grid and
then exploit conjugacy to get it in closed form.


In [ ]:
# Grid-based Bayes: unknown success probability p in [0,1].
# Prior: uniform (a special case of Beta(1,1)). Likelihood: Binomial(n,p).
a, b = 1.0, 1.0          # prior Beta(a,b)
n, k = 20, 14            # observe 14 successes in 20 trials

grid = np.linspace(1e-4, 1 - 1e-4, 2000)

prior = stats.beta(a, b).pdf(grid)
likelihood = stats.binom.pmf(k, n, grid)            # f(x | p)
unnorm = prior * likelihood                          # numerator
marginal = np.trapz(unnorm, grid)                    # denominator f(x)
posterior_grid = unnorm / marginal                  # normalized posterior

print(f"marginal likelihood f(x) = {marginal:.6f}")
print(f"integral of posterior    = {np.trapz(posterior_grid, grid):.6f}  (should be 1)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grid, prior, label="prior Beta(1,1)", lw=2)
ax.plot(grid, likelihood / np.trapz(likelihood, grid), label="likelihood (scaled)", lw=2)
ax.plot(grid, posterior_grid, label="posterior (grid)", lw=2, color="crimson")
ax.set_xlabel("p"); ax.set_ylabel("density")
ax.set_title(f"Bayes' rule on a grid: Beta(1,1) prior, k={k} successes in n={n}")
ax.legend()
plt.show()


---
## Part 2 — Conjugate priors: the Beta-Binomial model

A **conjugate prior** is one whose posterior lies in the *same family* as the
prior. For the Binomial likelihood with success probability \(p\),

$$
f(x \mid p) \propto p^{x}(1-p)^{n-x},
$$

the Beta prior \(p \sim \text{Beta}(\alpha,\beta)\),

$$
\pi(p) \propto p^{\alpha-1}(1-p)^{\beta-1},
$$

is conjugate. Multiplying,

$$
\pi(p \mid x) \propto p^{x+\alpha-1}(1-p)^{n-x+\beta-1},
$$

so the posterior is

$$
p \mid x \sim \text{Beta}(\alpha + x,\ \beta + n - x).
$$

The update rule is beautifully simple: **add the data to the prior counts**.
- prior pseudo-counts: \(\alpha\) successes, \(\beta\) failures
- data: \(x\) successes, \(n-x\) failures
- posterior pseudo-counts: \(\alpha+x\) successes, \(\beta+n-x\) failures

We verify that the grid posterior from Part 1 exactly matches the
\(\text{Beta}(\alpha+x,\beta+n-x)\) density.


In [ ]:
# Closed-form conjugate update vs. the grid posterior
a, b = 1.0, 1.0
n, k = 20, 14

post_a = a + k
post_b = b + (n - k)
print(f"prior     Beta({a},{b})")
print(f"data      k={k} successes in n={n} trials")
print(f"posterior Beta({post_a},{post_b})")

post_closed = stats.beta(post_a, post_b)

# Compare closed form to grid posterior from Part 1
grid = np.linspace(1e-4, 1 - 1e-4, 2000)
prior = stats.beta(a, b).pdf(grid)
likelihood = stats.binom.pmf(k, n, grid)
posterior_grid = prior * likelihood / np.trapz(prior * likelihood, grid)

max_err = np.max(np.abs(posterior_grid - post_closed.pdf(grid)))
print(f"max |grid - closed form| = {max_err:.2e}  (should be ~0)")
assert np.isclose(max_err, 0, atol=1e-6)
print("Conjugate update verified ✔")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grid, stats.beta(a, b).pdf(grid), label=f"prior Beta({a},{b})", lw=2)
ax.plot(grid, likelihood / np.trapz(likelihood, grid), label="likelihood (scaled)", lw=2)
ax.plot(grid, post_closed.pdf(grid), label=f"posterior Beta({post_a},{post_b})", lw=2, color="crimson")
ax.set_xlabel("p"); ax.set_ylabel("density")
ax.set_title("Beta-Binomial conjugate update")
ax.legend()
plt.show()


---
## Part 3 — Posterior mean and credible intervals

A natural Bayesian **point estimate** is the posterior mean
\(E[\theta \mid x]\). For the Beta-Binomial model,

$$
E[p \mid x] = \frac{\alpha + x}{\alpha + \beta + n}.
$$

This is a **weighted average** of the prior mean \(\alpha/(\alpha+\beta)\) and
the sample proportion \(x/n\), with weights determined by the prior strength
\(\alpha+\beta\) versus the data size \(n\):
- weak prior (\(\alpha+\beta \ll n\)) \(\Rightarrow\) posterior mean \(\approx x/n\)
- strong prior (\(\alpha+\beta \gg n\)) \(\Rightarrow\) posterior mean \(\approx\) prior mean

A \(95\%\) **credible interval** \((a,b)\) satisfies
\(P(a < p < b \mid x) = 0.95\). Unlike a frequentist confidence interval, this
is a direct probability statement *about the parameter*. We use the
posterior's equal-tailed quantiles: \(a = q_{0.025}\), \(b = q_{0.975}\).


In [ ]:
# Posterior mean and 95% credible interval for the Beta-Binomial model
a, b = 2.0, 2.0          # prior Beta(2,2) - mild prior toward 0.5
n, k = 30, 21            # observe 21 successes in 30 trials

post_a = a + k
post_b = b + (n - k)
post = stats.beta(post_a, post_b)

post_mean = (a + k) / (a + b + n)           # closed form
prior_mean = a / (a + b)
sample_prop = k / n

print(f"prior mean      = {prior_mean:.4f}")
print(f"sample prop x/n = {sample_prop:.4f}")
print(f"posterior mean  = {post_mean:.4f}  (closed form)")
print(f"posterior mean  = {post.mean():.4f}  (scipy)")
assert np.isclose(post_mean, post.mean())
print("Posterior mean formula verified ✔")

ci_low, ci_high = post.ppf(0.025), post.ppf(0.975)
print(f"95% credible interval = ({ci_low:.4f}, {ci_high:.4f})")
print(f"posterior mass inside  = {post.cdf(ci_high) - post.cdf(ci_low):.4f}  (should be 0.95)")

fig, ax = plt.subplots(figsize=(9, 5))
gs = np.linspace(1e-3, 1 - 1e-3, 1000)
ax.plot(gs, post.pdf(gs), color="crimson", lw=2, label=f"posterior Beta({post_a},{post_b})")
ax.axvline(post_mean, color="black", ls="--", label=f"posterior mean = {post_mean:.3f}")
ax.axvline(ci_low, color="grey", ls=":", label=f"95% CI = ({ci_low:.3f}, {ci_high:.3f})")
ax.axvline(ci_high, color="grey", ls=":")
ax.fill_between(gs[(gs >= ci_low) & (gs <= ci_high)],
                post.pdf(gs[(gs >= ci_low) & (gs <= ci_high)]),
                alpha=0.25, color="crimson")
ax.set_xlabel("p"); ax.set_ylabel("density")
ax.set_title("Beta-Binomial posterior: mean and 95% credible interval")
ax.legend()
plt.show()


---
## Part 4 — Normal-Normal conjugate update for a mean (known variance)

Conjugacy is not unique to the Beta-Binomial. For a normal likelihood with
**known** variance \(\sigma^2\),

$$
X_1,\dots,X_n \mid \mu \sim \mathcal{N}(\mu, \sigma^2),
$$

a normal prior \(\mu \sim \mathcal{N}(\mu_0, \tau_0^2)\) is conjugate. The
posterior is again normal,

$$
\mu \mid x \sim \mathcal{N}(\mu_n, \tau_n^2),
$$

with precision (inverse variance) adding:

$$
\frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2},
\qquad
\mu_n = \tau_n^2 \left( \frac{\mu_0}{\tau_0^2} + \frac{\bar{x}}{\sigma^2/n} \cdot n \right)
      = \tau_n^2 \left( \frac{\mu_0}{\tau_0^2} + \frac{n\,\bar{x}}{\sigma^2} \right).
$$

Equivalently the posterior mean is a precision-weighted average of the prior
mean and the sample mean:

$$
\mu_n = \frac{\tau_0^{-2}\,\mu_0 + n\,\sigma^{-2}\,\bar{x}}
             {\tau_0^{-2} + n\,\sigma^{-2}}.
$$

More data (larger \(n\) or smaller \(\sigma^2\)) pulls the posterior toward
\(\bar{x}\); a tighter prior (smaller \(\tau_0^2\)) keeps it near \(\mu_0\).

We simulate data from a known \(\mu\), apply the update, and plot the prior,
likelihood (as a function of \(\mu\)), and posterior.


In [ ]:
# Normal-Normal conjugate update for a mean with known variance
mu0, tau0_sq = 0.0, 4.0      # prior: N(0, 4)  (weak, sd = 2)
sigma_sq = 1.0               # known data variance
true_mu = 1.5
n = 25

rng = np.random.default_rng(42)
x = rng.normal(true_mu, np.sqrt(sigma_sq), size=n)
xbar = x.mean()

# Conjugate update (precision form)
prec_prior = 1.0 / tau0_sq
prec_data  = n / sigma_sq
post_prec  = prec_prior + prec_data
tau_n_sq   = 1.0 / post_prec
mu_n       = tau_n_sq * (prec_prior * mu0 + prec_data * xbar)

post = stats.norm(mu_n, np.sqrt(tau_n_sq))
print(f"prior      N({mu0}, {tau0_sq})")
print(f"data       n={n}, xbar={xbar:.4f}, true mu={true_mu}")
print(f"posterior  N({mu_n:.4f}, {tau_n_sq:.4f})")
print(f"posterior mean (formula)  = {mu_n:.6f}")
print(f"posterior mean (precision) = {post.mean():.6f}")
assert np.isclose(mu_n, post.mean())
print("Normal-Normal conjugate update verified ✔")

# Plot prior, (scaled) likelihood as a function of mu, posterior
mus = np.linspace(-2.5, 3.5, 1000)
prior_pdf = stats.norm(mu0, np.sqrt(tau0_sq)).pdf(mus)
like_as_mu = np.exp(-0.5 * np.sum((x[None, :] - mus[:, None])**2, axis=1) / sigma_sq)
like_as_mu = like_as_mu / np.trapz(like_as_mu, mus)   # normalize for display

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mus, prior_pdf, lw=2, label=f"prior N({mu0},{tau0_sq})")
ax.plot(mus, like_as_mu, lw=2, label="likelihood (scaled)")
ax.plot(mus, post.pdf(mus), lw=2, color="crimson",
        label=f"posterior N({mu_n:.2f},{tau_n_sq:.2f})")
ax.axvline(xbar, color="grey", ls=":", label=f"xbar = {xbar:.3f}")
ax.set_xlabel(r"$\mu$"); ax.set_ylabel("density")
ax.set_title("Normal-Normal conjugate update for a mean (known variance)")
ax.legend()
plt.show()


---
## Part 5 — 95% credible interval and coverage by simulation

For the Normal posterior from Part 4, the 95% credible interval is

$$
\mu_n \pm 1.96\,\sqrt{\tau_n^2}.
$$

Bayesian credible intervals have a clean **coverage** property: if the model
is correctly specified, the posterior 95% interval contains the true
parameter in (about) 95% of repeated experiments. We check this by
simulation: for many replications we draw data from the true \(\mu\), form the
posterior interval, and record whether it contains the true \(\mu\). The
empirical coverage should be close to \(0.95\).


In [ ]:
# Coverage simulation for the 95% Normal-Normal credible interval
mu0, tau0_sq = 0.0, 4.0
sigma_sq = 1.0
true_mu = 1.5
n = 25
N_reps = 2000

rng = np.random.default_rng(123)
covered = np.zeros(N_reps, dtype=bool)
widths = np.empty(N_reps)

for r in range(N_reps):
    xx = rng.normal(true_mu, np.sqrt(sigma_sq), size=n)
    xbar = xx.mean()
    prec_prior = 1.0 / tau0_sq
    prec_data  = n / sigma_sq
    tau_n_sq   = 1.0 / (prec_prior + prec_data)
    mu_n       = tau_n_sq * (prec_prior * mu0 + prec_data * xbar)
    half = stats.norm.ppf(0.975) * np.sqrt(tau_n_sq)
    lo, hi = mu_n - half, mu_n + half
    covered[r] = (lo <= true_mu <= hi)
    widths[r] = hi - lo

coverage = covered.mean()
mean_width = widths.mean()
print(f"replications           = {N_reps}")
print(f"empirical coverage     = {coverage:.4f}  (target 0.95)")
print(f"mean interval width     = {mean_width:.4f}")
print(f"analytic posterior sd   = {np.sqrt(1.0/(1/tau0_sq + n/sigma_sq)):.4f}")

# Coverage should be within ~2 SE of 0.95
se = np.sqrt(0.95 * 0.05 / N_reps)
print(f"std error of coverage   = {se:.4f}")
print(f"within 2 SE of 0.95?     {abs(coverage - 0.95) < 2 * se}")

fig, ax = plt.subplots(figsize=(9, 5))
first = 60
for r in range(first):
    xx = rng.normal(true_mu, np.sqrt(sigma_sq), size=n)
    xbar = xx.mean()
    tau_n_sq = 1.0 / (1.0/tau0_sq + n/sigma_sq)
    mu_n = tau_n_sq * (1.0/tau0_sq * mu0 + n/sigma_sq * xbar)
    half = stats.norm.ppf(0.975) * np.sqrt(tau_n_sq)
    color = "steelblue" if (mu_n - half) <= true_mu <= (mu_n + half) else "crimson"
    ax.plot([mu_n - half, mu_n + half], [r, r], color=color, lw=1)
ax.axvline(true_mu, color="black", lw=2, label=f"true mu = {true_mu}")
ax.set_ylabel("replication"); ax.set_xlabel(r"$\mu$")
ax.set_title(f"95% credible intervals for first {first} replications "
             f"(coverage = {coverage:.3f})")
ax.legend()
plt.show()


---
**Your turn:**

- Re-run Part 3 with a **strong prior** \(\text{Beta}(50,50)\) and the same data
  \((n,k)=(30,21)\). How does the posterior mean shift? Compare its weight on
  the prior mean vs. the sample proportion.
- In Part 4, increase \(n\) from \(25\) to \(200\). How does the posterior
  variance \(\tau_n^2\) change? Does the posterior mean move toward \(\bar{x}\)?
- In Part 5, change the prior variance \(\tau_0^2\) to \(0.01\) (a very
  confident but *wrong* prior \(\mu_0 = 0\) when \(\mu_{\text{true}} = 1.5\)).
  What happens to coverage? Why?
- Replace the equal-tailed interval in Part 3 with the **HPD** (highest
  posterior density) interval for the Beta posterior — use the fact that the
  Beta density is unimodal and find the narrowest interval of mass \(0.95\)
  by a numerical search.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Suppose \(p \sim \text{Beta}(2,5)\) and we observe \(x=8\) successes in
\(n=12\) trials. State the posterior distribution, compute the posterior mean,
and give a 95% credible interval.

2. Show numerically that the Beta-Binomial posterior mean
\((\alpha+x)/(\alpha+\beta+n)\) is a precision-weighted average of the prior
mean \(\alpha/(\alpha+\beta)\) and the sample proportion \(x/n\). Specifically,
write it as
\(\frac{w_{\text{prior}}\,\mu_{\text{prior}} + w_{\text{data}}\,\hat{p}}{w_{\text{prior}}+w_{\text{data}}}\)
and identify the weights. Verify the algebra with NumPy for
\((\alpha,\beta)=(3,7),\ (n,x)=(40,25)\).

3. For the Normal-Normal model with \(\mu_0=5,\ \tau_0^2=2,\ \sigma^2=3,\ n=10\)
and observed \(\bar{x}=4.2\), compute \(\mu_n\) and \(\tau_n^2\) from the
conjugate update. Then compute the 95% credible interval and check whether it
covers \(\bar{x}\) (the MLE).

4. Run a coverage simulation for the **Beta-Binomial** 95% credible
interval. Use prior \(\text{Beta}(1,1)\), true \(p=0.3\), \(n=50\), and
2000 replications. Report the empirical coverage. Is it close to \(0.95\)?
(Hint: the posterior is \(\text{Beta}(1+k,\ 1+n-k)\); the equal-tailed
95% interval is `beta.ppf([0.025, 0.975])`.)

5. Write a function `beta_binomial_update(a, b, n, k)` that returns the
posterior distribution (as a `scipy.stats` frozen distribution), the posterior
mean, and the 95% credible interval. Apply it to three datasets with the same
prior \(\text{Beta}(2,2)\): \((n,k)=(10,7),\ (100,70),\ (1000,700)\). How
does the posterior width shrink as \(n\) grows?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Posterior \(\text{Beta}(2+8,\ 5+12-8) = \text{Beta}(10,9)\).
Posterior mean \(= 10/(10+9) = 0.5263\). A 95% equal-tailed interval:

```python
from scipy import stats
post = stats.beta(10, 9)
print(post.mean())                       # 0.5263
print(post.ppf([0.025, 0.975]))           # [0.3234, 0.7245]
```

**2.** Let \(\mu_p=\alpha/(\alpha+\beta)\), \(\hat p = x/n\). Then
\((\alpha+x)/(\alpha+\beta+n) = \frac{(\alpha+\beta)\mu_p + n\hat p}{\alpha+\beta+n}\),
so \(w_{\text{prior}}=\alpha+\beta\) (prior pseudo-count) and \(w_{\text{data}}=n\).

```python
import numpy as np
a, b, n, x = 3, 7, 40, 25
mu_p, phat = a/(a+b), x/n
post_mean = (a + x) / (a + b + n)
weighted  = ((a+b)*mu_p + n*phat) / (a+b+n)
print(np.isclose(post_mean, weighted))   # True
```

**3.** \(\tau_n^2 = 1/(1/2 + 10/3) = 0.2836\);
\(\mu_n = 0.2836 \cdot (5/2 + 10 \cdot 4.2/3) = 4.284\).
95% CI \(\approx 4.284 \pm 1.96\sqrt{0.2836} = (3.241,\ 5.327)\). It covers
\(\bar{x}=4.2\).

```python
import numpy as np
from scipy import stats
mu0, tau0sq, sig2, n, xbar = 5, 2, 3, 10, 4.2
tau_n_sq = 1.0/(1/tau0sq + n/sig2)
mu_n = tau_n_sq * (mu0/tau0sq + n*xbar/sig2)
lo, hi = mu_n + stats.norm.ppf([0.025, 0.975]) * np.sqrt(tau_n_sq)
print(mu_n, (lo, hi), lo <= xbar <= hi)   # 4.284 ..., True
```

**4.** With a \(\text{Beta}(1,1)\) prior the posterior is \(\text{Beta}(1+k,\ 1+n-k)\),
so the interval is exactly `beta(1+k,1+n-k).ppf([0.025,0.975])`. Coverage:

```python
import numpy as np
from scipy import stats
rng = np.random.default_rng(0)
p_true, n, N = 0.3, 50, 2000
cov = 0
for _ in range(N):
    k = rng.binomial(n, p_true)
    lo, hi = stats.beta(1+k, 1+n-k).ppf([0.025, 0.975])
    cov += (lo <= p_true <= hi)
print(cov / N)   # ~0.95
```

**5.** A reusable update routine; note the posterior width scales like
\(1/\sqrt{n}\) (information grows linearly).

```python
from scipy import stats

def beta_binomial_update(a, b, n, k):
    post = stats.beta(a + k, b + n - k)
    mean = post.mean()
    lo, hi = post.ppf([0.025, 0.975])
    return post, mean, (lo, hi)

for (n, k) in [(10, 7), (100, 70), (1000, 700)]:
    _, m, (lo, hi) = beta_binomial_update(2, 2, n, k)
    print(f"n={n:5d}  mean={m:.4f}  width={hi-lo:.4f}")
# n=   10  mean=0.7500  width=0.5330
# n=  100  mean=0.7059  width=0.1769
# n= 1000  mean=0.7006  width=0.0567
```


</details>
